# LeakAudit - 04 - Time-of-day confound on D2 (E6)

**Predict leak vs no-leak from the recording time alone - the audio is never opened.**

- Every Hong Kong leak clip carrying a timestamp was recorded between 03:30 and 04:15; no-leak clips span 03:30 to 22:00, with roughly a third also recorded before dawn. The asymmetry is therefore that the leak class has no temporal diversity at all, not that the two classes sit at opposite ends of the day.
- The dataset description states that leak signals were collected when leakage was reported and no-leak signals after the site had been repaired. A similar procedure is described for a multi-city dataset (Water Research X, 2025), so we treat this as a recurring evaluation-validity risk across heterogeneous datasets rather than an established field-wide practice.
- If the clock alone separates the classes, any acoustic score is entangled with this confound.
- This defines the *signal-free baseline* (SFB) we recommend every study report.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_score, GroupKFold

M2 = io_utils.scan_d2()
print(f"D2 files: {len(M2)}   loggers: {M2.logger.nunique()}")
print(M2.groupby(["sensor", "folder"]).size().to_string())

D2 files: 160   loggers: 52
sensor               folder         
Hydrophones          Leak               40
                     No-Leak            40
MEMS Accelerometers  Leak               10
                     No-Leak            10
Noise Loggers        Leak-Metal         15
                     Leak-NonMetal      15
                     NoLeak-Metal       15
                     NoLeak-NonMetal    15


## Recording time versus label

In [3]:
sub = M2[(M2.ext == ".wav") & (M2.label.isin(["leak", "noleak"])) & M2.hhmm.notna()].copy()
sub["hour"] = sub.hhmm.str[:2].astype(int) + sub.hhmm.str[2:].astype(int) / 60.0
print(pd.crosstab(sub.label, sub.hhmm).to_string())
print(f"\nmean hour  leak = {sub[sub.label=='leak'].hour.mean():.2f} h   "
      f"noleak = {sub[sub.label=='noleak'].hour.mean():.2f} h")

hhmm    0330  0345  0400  0415  0430  0630  0730  0830  0930  1030  1145  1248  1445  1538  1800  1900  2000  2100  2200
label                                                                                                                   
leak      14    14    10    10     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0
noleak     4     4     5     5     4     2     2     2     2     2    24     1     2     1     2     2     2     2     2

mean hour  leak = 3.83 h   noleak = 10.19 h


## Signal-free baseline

A depth-1 decision stump on the single feature *hour*, scored two ways:
leave-one-out, and logger-independent (so the stump cannot memorise a logger).

In [4]:
Xh = sub[["hour"]].to_numpy()
yh = sub.label.to_numpy()
stump = DecisionTreeClassifier(max_depth=1, random_state=0)

acc_loo = cross_val_score(stump, Xh, yh, cv=LeaveOneOut()).mean()
base_h  = pd.Series(yh).value_counts(normalize=True).max()
acc_grp = np.nan
if sub.logger.nunique() >= 4:
    gk = GroupKFold(n_splits=min(5, sub.logger.nunique()))
    acc_grp = cross_val_score(stump, Xh, yh, cv=gk, groups=sub.logger).mean()

print(f"time-only stump (leave-one-out)   : {acc_loo:.4f}")
print(f"time-only stump (logger-independent): {acc_grp:.4f}")
print(f"majority-class baseline           : {base_h:.4f}")

res = dict(n=len(sub), acc_hour_loo=float(acc_loo),
           acc_hour_grouped=float(acc_grp) if acc_grp == acc_grp else None,
           baseline=float(base_h),
           mean_hour_leak=float(sub[sub.label == 'leak'].hour.mean()),
           mean_hour_noleak=float(sub[sub.label == 'noleak'].hour.mean()))
la.save_json(res, "E6_time_confound_D2.json")
print("\n-> the gap above the baseline is the confound the audio cannot be separated from.")

time-only stump (leave-one-out)   : 0.8475
time-only stump (logger-independent): 0.8471
majority-class baseline           : 0.5932

-> the gap above the baseline is the confound the audio cannot be separated from.
